# S1 — Pipeline complet : résistance du béton (génie civil)

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/inphb/ic1/notebooks/IC1_S1_pipeline_beton.ipynb)
> Données : dossier `IC1_TP_donnees`. Bibliothèques : pandas, numpy, matplotlib, scikit-learn.

In [ ]:
# Configuration : Colab ou Jupyter local. Exécutez cette cellule en premier.
import os, shutil, urllib.request
DONNEES = ['beton_8_constituants.csv']
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/inphb/ic1/data/'
for f in DONNEES:
    if os.path.exists(f):
        continue
    if os.path.exists(os.path.join('..', 'data', f)):
        shutil.copy(os.path.join('..', 'data', f), f)   # kit hors ligne / dépôt cloné
    else:
        urllib.request.urlretrieve(URL + f, f)            # Colab : téléchargement
    print('OK', f)


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold, cross_val_score, cross_val_predict, learning_curve, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score, ConfusionMatrixDisplay
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
b = pd.read_csv('beton_8_constituants.csv'); print(b.isna().sum())
X, y = b.drop(columns='resistance_MPa'), b['resistance_MPa']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)

## 1. Trois modèles dans des pipelines (aucune fuite)

In [ ]:
modeles = {
 'lin': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LinearRegression()),
 'ridge': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), Ridge(alpha=10)),
 'rf': make_pipeline(SimpleImputer(strategy='median'), RandomForestRegressor(400, min_samples_leaf=2, random_state=0))}
for nom, m in modeles.items():
    m.fit(X_tr, y_tr); p = m.predict(X_te)
    print(nom, round(mean_absolute_error(y_te, p), 2), round(r2_score(y_te, p), 3))

## 2. Courbe d’apprentissage : faut-il plus d’essais ?

In [ ]:
sz, tr, va = learning_curve(modeles['rf'], X, y, cv=5, scoring='r2', train_sizes=np.linspace(.1, 1, 6))
plt.plot(sz, tr.mean(1), 'o-', label='entraînement'); plt.plot(sz, va.mean(1), 'o-', label='validation'); plt.legend(); plt.show()

## Bonus
Imputer le superplastifiant par **0** (`SimpleImputer(strategy='constant', fill_value=0)`) et comparer. Quelle hypothèse est la plus réaliste ?